# QLoRA fine-tune Qwen2.5-3B trên Kaggle (chạy nền ~12h)
**Settings** (panel phải): Accelerator = **GPU T4 x2** (hoặc P100) · Internet = **On**.

**Chạy NỀN:** **Save Version → Save & Run All (Commit)** → headless tới ~12h, adapter ở `/kaggle/working/qwen-lora` (tab Output).

Self-host ≤9B, KHÔNG API ngoài. Data = **distill frontier (nhãn thật, phân phối test) + synthetic (phủ format)**.
Lưu ý: synthetic-only từng chỉ đạt dev FINAL **0.2832 < rule 0.4515** → distill là tín hiệu chính. T4 ~7-8h/2 epoch.

In [ ]:
# 1) Lấy code — nhánh Duckun (P2, builder chính). Main đang cũ, KHÔNG dùng.
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git fetch -q origin Duckun && git checkout -q Duckun && git pull -q origin Duckun
!git log --oneline -1

In [ ]:
# 2) Cài env — KHÔNG động vào numpy/transformers/torch (dùng bản Kaggle ship sẵn).
#    Chỉ update bitsandbytes/peft/accelerate.
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch, transformers, bitsandbytes as bnb, peft
print('numpy', numpy.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bnb.__version__, '| peft', peft.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

In [ ]:
# 2b) Dựng SFT data từ nguồn ĐÃ CÓ trong repo (raw_extract frontier + synthetic) — reproducible.
!python src/datagen/distill_frontier.py     # raw_extract -> train_sft_distill.jsonl (124 chunk-aligned)
!python src/datagen/mix_sft.py               # + synthetic -> train_sft_mixed_t4.jsonl (672: distill 55%/synth 45%)

In [ ]:
# 3) SMOKE-TRAIN vài bước (kiểm không OOM ở ĐÚNG max_len train = 2560). 1 GPU cho ổn định.
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # OOM -> 'Qwen/Qwen2.5-1.5B-Instruct'
DATA  = 'data/silver/frontier_ref/train_sft_mixed_t4.jsonl'   # 672 mẫu: distill 55% + synthetic 45%
!python scripts/train_qlora.py --data {DATA} --model {MODEL} --out /kaggle/working/smoke-lora --epochs 0.05 --max-len 2560 --seed 42

In [ ]:
# 4) TRAIN — distill-nặng, ~7-8h/2 epoch trên T4 (672 mẫu; max_len 2560 giữ TRỌN 124 distill dài).
#    KHÔNG dùng --max-samples (nó cắt theo thứ tự file -> cắt trúng distill); file đã đúng cỡ.
ADAPTER = '/kaggle/working/qwen-lora'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
DATA  = 'data/silver/frontier_ref/train_sft_mixed_t4.jsonl'
!python scripts/train_qlora.py --data {DATA} --model {MODEL} --out {ADAPTER} --epochs 2 --batch 1 --grad-accum 8 --lr 2e-4 --max-len 2560 --seed 42
!ls -la {ADAPTER}

In [ ]:
# 5) Trỏ config sang adapter + backend llm (inference tự bỏ few-shot khi có adapter)
import yaml
ADAPTER = '/kaggle/working/qwen-lora'
cfg = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
cfg['extract']['backend'] = 'llm'
cfg['extract']['lora_adapter'] = ADAPTER
cfg['extract']['llm_model'] = 'Qwen/Qwen2.5-3B-Instruct'
yaml.safe_dump(cfg, open('configs/config.yaml', 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print('lora_adapter =', cfg['extract']['lora_adapter'])

In [ ]:
# 6) GO/NO-GO — đo trên 60 file dev có gold (METRIC CHÍNH THỨC).
#    Mốc so: rule=0.4515 | synthetic-only QLoRA=0.2832 | frontier-ceiling=0.5453 (offline).
#    QLoRA distill PHẢI vượt rule (0.4515) mới đáng nộp.
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
!python src/pipeline.py --input devset/input --output dev_pred --backend llm
!python src/eval/official_scorer.py --pred dev_pred --gold data/dev/gold

In [ ]:
# 7) (Khi hài lòng) chạy full 100 file + đóng gói. output.zip + qwen-lora ở /kaggle/working (tab Output).
!python src/pipeline.py --input data/test/input --output /kaggle/working/vmr_output --backend llm
!python scripts/package_submission.py --output /kaggle/working/vmr_output --input data/test/input --n 100
!cp output.zip /kaggle/working/output.zip 2>/dev/null; ls -la /kaggle/working/*.zip